In [ ]:
from IPython.display import clear_output

In [ ]:
# %pip install gensim nltk tqdm scikit-learn

%pip install datasets

clear_output()

# Content

In this demo, we will train a word2vec model on custom data. We will use IMDB dataset as our training data

a word2vec model is another way to convert strings(or words) to numerical represntation, so those vectors can then be used to perform some task or train another model.

A word2vec uses Continuous bag of words technique to learn embeddings of words it sees during training, and then saves them. After training, we can infer those vectors by providing the word. We will not delve into details about contiuous bag of words (CBOW) here.

The special thing about word2vec vectors (which is not found in TF-IDF or word to index) is that the distance in word2vec vectors (between each other) also represent information about how similar the words are to each other

for example, the vectors of words "cat" and "dog" will have a higher similarity (or lower distance) compared to the vectors of words "cat" and "building"


In [ ]:
import nltk
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset


nltk.download('punkt')

clear_output()

## Preparing the data

In [ ]:
train_data = load_dataset('imdb', split='train')['text']
train_data = train_data[:500]  # shorten the data because all 25k rows take too long to train

In [ ]:
tokenized_train_data = [word_tokenize(text.lower()) for text in tqdm(train_data, desc='Tokenizing data')]

## Train the model

In [ ]:
# with tqdm(total=len(tokenized_train_data), desc='Training model') as pbar:
model = Word2Vec(tokenized_train_data, vector_size=128, window=5, min_count=1, workers=4)

In [ ]:
word_vectors = model.wv

## Try the model

In [ ]:
test_words = ['cat', 'dog', 'building', 'movie', 'action', 'comedy', 'fight', 'plot', 'laugh']
test_word_vectors = [word_vectors[word] for word in test_words]

test_words_similarity_map = {}  # map each word to all other words where the other words list is sorted according to similarity (most similar first)

for i, word in enumerate(test_words):

    remaining_words = test_words[:i]+test_words[i+1:]
    remaining_word_vectors = test_word_vectors[:i]+test_word_vectors[i+1:]

    word_to_remaining_cosine_sims = cosine_similarity(test_word_vectors[i].reshape(1, -1), remaining_word_vectors)[0]
    sorting_order = word_to_remaining_cosine_sims.argsort()[::-1]  # [::-1] because argsort returns lowest to highest and we want highest to lowest because higher cosine sim is more similar word
    ordered_remaining_words = [remaining_words[j] for j in sorting_order]  # ordered according to similarity with the word(in loop iteration)

    test_words_similarity_map[word] = list(zip(ordered_remaining_words, word_to_remaining_cosine_sims[sorting_order]))

In [ ]:
test_words_similarity_map['cat']

In [ ]:
test_words_similarity_map['action']

In [ ]:
test_words_similarity_map['comedy']

In [ ]:
test_words_similarity_map['laugh']

## Finidng similar words

Since the vectors now represent a dimension of similarity, word2vec also allows us to find similar words to a word (much like we did above but from the whole corpus instead of a selective words)

word2vec also performs reasonably well for doing things like:

king_vec - man_vec + woman_vec = queen_vec (a famous example). However for good results, do note that the model needs to be trained on siginificant data. Our model is trained on a very small data (relatively) to do things like this

In [ ]:
word_vectors.most_similar('movie')

In [ ]:
word_vectors.most_similar('mother')